# Agregação e envio da Vacinação para o Supabase

**v2 — reescrito.** As tabelas do SINAN e do SIH só existem no Supabase, não no Databricks — então os cruzamentos (Cards 02, 03 e 04) não podem ser feitos aqui via Spark. Este notebook faz só a parte que depende da nossa Silver (`VACINACAO_DENGUE_SP`): agrega por período/município e por período/município/faixa etária, e envia pro Supabase. Os cruzamentos em si rodam como SQL puro direto no Supabase (`cruzamentos_vacinacao.sql`), depois que este notebook rodar.

In [0]:
%pip install --upgrade typing_extensions supabase
dbutils.library.restartPython()

In [0]:
import typing_extensions
from supabase import create_client, Client

print("Importação concluída com sucesso!")

In [0]:
CATALOGO = "fiap"

### Conexão com o Supabase

In [0]:
SUPABASE_URL = dbutils.secrets.get(scope="sus_predict", key="supabase_url")
SUPABASE_KEY = dbutils.secrets.get(scope="sus_predict", key="supabase_key")

supabase: Client = create_client(SUPABASE_URL, SUPABASE_KEY)
print("Conectado ao Supabase")

In [0]:
from decimal import Decimal
import pandas as pd
import numpy as np
import math
import json

def enviar_para_supabase(
    tabela_spark,
    nome_tabela_supabase,
    truncate=True
):

    try:

        df_pandas = tabela_spark.toPandas()

        for col in df_pandas.columns:

            if pd.api.types.is_datetime64_any_dtype(df_pandas[col]):
                df_pandas[col] = df_pandas[col].astype(str)

            df_pandas[col] = df_pandas[col].apply(
                lambda x: str(x)
                if hasattr(x, "isoformat")
                else x
            )

        for col in df_pandas.columns:
            df_pandas[col] = df_pandas[col].apply(
                lambda x: float(x)
                if isinstance(x, Decimal)
                else x
            )

        df_pandas.columns = [
            col.lower()
            for col in df_pandas.columns
        ]

        dados = df_pandas.to_dict("records")

        for linha in dados:

            for chave, valor in list(linha.items()):

                if isinstance(valor, np.integer):
                    linha[chave] = int(valor)

                elif isinstance(valor, np.floating):

                    if np.isnan(valor) or np.isinf(valor):
                        linha[chave] = None
                    else:
                        linha[chave] = float(valor)

                elif isinstance(valor, float):

                    if math.isnan(valor) or math.isinf(valor):
                        linha[chave] = None

        json.dumps(dados, allow_nan=False)

        print(f"\nenviando {nome_tabela_supabase}")
        print(f"total {len(dados):,.0f} registros")

        if truncate:

            try:

                primeira_coluna = df_pandas.columns[0]

                supabase.table(nome_tabela_supabase) \
                    .delete() \
                    .not_.is_(primeira_coluna, "null") \
                    .execute()

                print(f"tabela {nome_tabela_supabase} limpa")

            except Exception as e:
                print(f"não foi possivel limpar: {e}")

        chunk_size = 1000
        total_inserido = 0

        for i in range(0, len(dados), chunk_size):

            chunk = dados[i:i + chunk_size]

            supabase.table(nome_tabela_supabase) \
                .insert(chunk) \
                .execute()

            total_inserido += len(chunk)

            print(
                f"inseridos {len(chunk):,.0f} registros "
                f"({total_inserido:,.0f}/{len(dados):,.0f})"
            )

        print(f"{nome_tabela_supabase} enviado com sucesso\n")

        return True

    except Exception as e:

        print(f"ERRO ao enviar {nome_tabela_supabase}: {e}")
        return False

# Card 01 — `vacinacao_dengue_municipios`

Agrega a Silver `VACINACAO_DENGUE_SP` por município, nas mesmas janelas de período usadas em todo o resto do projeto (Trimestre, Semestre, 12 Meses, 3 Anos, 5 Anos), tomando como referência o mês mais recente disponível na própria base de vacinação.

In [0]:
%sql
CREATE OR REPLACE TABLE fiap.silver.VACINACAO_DENGUE_MUNICIPIOS AS

WITH referencia AS (
    SELECT MAX(dt_vacina) AS dt_ref
    FROM fiap.silver.VACINACAO_DENGUE_SP
),

periodos AS (
    SELECT 'Trimestre' AS periodo, ADD_MONTHS(dt_ref, -2) AS dt_inicio, dt_ref AS dt_fim FROM referencia
    UNION ALL
    SELECT 'Semestre', ADD_MONTHS(dt_ref, -5), dt_ref FROM referencia
    UNION ALL
    SELECT '12 Meses', ADD_MONTHS(dt_ref, -11), dt_ref FROM referencia
    UNION ALL
    SELECT '3 Anos', ADD_MONTHS(dt_ref, -35), dt_ref FROM referencia
    UNION ALL
    SELECT '5 Anos', ADD_MONTHS(dt_ref, -59), dt_ref FROM referencia
),

base AS (
    SELECT
        p.periodo,
        v.cod_ibge_municipio,
        v.dt_vacina
    FROM fiap.silver.VACINACAO_DENGUE_SP v
    CROSS JOIN periodos p
    WHERE v.dt_vacina BETWEEN p.dt_inicio AND p.dt_fim
)

SELECT
    'Dengue' AS id_agravo,
    periodo,
    cod_ibge_municipio,
    COUNT(*) AS doses_aplicadas,
    CURRENT_TIMESTAMP() AS data_referencia
FROM base
GROUP BY periodo, cod_ibge_municipio
ORDER BY periodo, cod_ibge_municipio

# Card 04 — `vacinacao_dengue_faixa_etaria_doses`

Mesma lógica acima, mas agregando também por faixa etária (padrão do projeto: 0–9, 10–19, 20–39, 40–59, 60–79, 80+), pra alimentar o cruzamento com `sinan_dengue_municipios_faixa_etaria` no Supabase (Card 04 da Tela Vacinação).

In [0]:
%sql
CREATE OR REPLACE TABLE fiap.silver.VACINACAO_DENGUE_FAIXA_ETARIA_DOSES AS

WITH referencia AS (
    SELECT MAX(dt_vacina) AS dt_ref
    FROM fiap.silver.VACINACAO_DENGUE_SP
),

periodos AS (
    SELECT 'Trimestre' AS periodo, ADD_MONTHS(dt_ref, -2) AS dt_inicio, dt_ref AS dt_fim FROM referencia
    UNION ALL
    SELECT 'Semestre', ADD_MONTHS(dt_ref, -5), dt_ref FROM referencia
    UNION ALL
    SELECT '12 Meses', ADD_MONTHS(dt_ref, -11), dt_ref FROM referencia
    UNION ALL
    SELECT '3 Anos', ADD_MONTHS(dt_ref, -35), dt_ref FROM referencia
    UNION ALL
    SELECT '5 Anos', ADD_MONTHS(dt_ref, -59), dt_ref FROM referencia
),

base AS (
    SELECT
        p.periodo,
        v.cod_ibge_municipio,
        CASE
            WHEN v.idade_paciente BETWEEN 0 AND 9   THEN '0-9'
            WHEN v.idade_paciente BETWEEN 10 AND 19 THEN '10-19'
            WHEN v.idade_paciente BETWEEN 20 AND 39 THEN '20-39'
            WHEN v.idade_paciente BETWEEN 40 AND 59 THEN '40-59'
            WHEN v.idade_paciente BETWEEN 60 AND 79 THEN '60-79'
            ELSE '80+'
        END AS faixa_etaria
    FROM fiap.silver.VACINACAO_DENGUE_SP v
    CROSS JOIN periodos p
    WHERE v.dt_vacina BETWEEN p.dt_inicio AND p.dt_fim
      AND v.idade_paciente IS NOT NULL
)

SELECT
    'Dengue' AS id_agravo,
    periodo,
    cod_ibge_municipio,
    faixa_etaria,
    COUNT(*) AS doses_aplicadas,
    CURRENT_TIMESTAMP() AS data_referencia
FROM base
GROUP BY periodo, cod_ibge_municipio, faixa_etaria
ORDER BY periodo, cod_ibge_municipio, faixa_etaria

# Envio ao Supabase

In [0]:
enviar_para_supabase(spark.table("fiap.silver.VACINACAO_DENGUE_MUNICIPIOS"), "vacinacao_dengue_municipios")

In [0]:
enviar_para_supabase(spark.table("fiap.silver.VACINACAO_DENGUE_FAIXA_ETARIA_DOSES"), "vacinacao_dengue_faixa_etaria_doses")

# Dimensão `ibge_sp`


In [0]:
from pyspark.sql.functions import current_timestamp

df_ibge_supabase = (
    spark.table("fiap.silver.IBGE_SP")
    .withColumnRenamed("COD_IBGE_COMPLETO", "cod_ibge_completo")
    .withColumnRenamed("COD_SUS", "cod_sus")
    .withColumnRenamed("NOME_MUNICIPIO", "nome_municipio")
    .withColumnRenamed("NOME_MICRORREGIAO", "nome_microrregiao")
    .withColumnRenamed("NOME_MESORREGIAO", "nome_mesorregiao")
    .withColumn("data_referencia", current_timestamp())
)

enviar_para_supabase(df_ibge_supabase, "ibge_sp")

# Próximo passo

Depois que as 3 tabelas acima (`vacinacao_dengue_municipios`, `vacinacao_dengue_faixa_etaria_doses`, `ibge_sp`) estiverem no Supabase, rodar o `cruzamentos_vacinacao.sql` no SQL Editor do Supabase — ele faz os cruzamentos com SINAN/SIH (Cards 02, 03 e 04) via SQL puro.